# NeuroObfuscator — Phase 3: QLoRA fine-tuning (Unsloth)

Обучение модели, генерирующей **план обфускации** (JSON, без seed) по JS-коду + AST-признакам.

**Датасет**: `data/final_v5_noseed/{train,val}.jsonl` — 6 840 train / 850 val (загрузить `final_v5_noseed.zip` в Google Drive).

## Железо и пресеты

| Пресет | GPU | Модель | Batch |
|---|---|---|---|
| `l4_7b` (основной) | **L4 22.5 GB** (Colab) | CodeLlama-7B-Instruct, 4-bit NF4 | 8 × 2 = 16 |
| `t4_3b` (fallback) | T4 15 GB и слабее | Qwen2.5-Coder-3B-Instruct, 4-bit | 8 × 2 = 16 |

Пресет выбирается **автоматически** по VRAM (порог 20 GB), можно задать вручную в ячейке конфига (`MANUAL_PRESET`).

Гиперпараметры: r=32, alpha=64, lr=2e-4, 3 эпохи, cosine, effective batch 16 (bs 8 × ga 2).

**Время на L4**: полный прогон с eval на всём val каждые 50 шагов — замер **~5.5 ч** (eval съедает ~2 ч). Перекалиброванная конфигурация (eval на 200-примерном подмножестве каждые 100 шагов, eval bs 16) — ожидается **~3–3.5 ч**. Нужно быстрее: `num_train_epochs=2` → ~2.5 ч.

In [ ]:
%pip install unsloth
# Если Colab предложит перезапустить runtime после установки — перезапустить и продолжить отсюда

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/neuroobfuscator/final_v5_noseed'  # поправить под свой путь в Drive
ADAPTER_OUT = '/content/drive/MyDrive/neuroobfuscator/adapters_v5'
!ls {DATA_DIR}

In [ ]:
import torch

# None = автодетект по VRAM; либо явно 'l4_7b' / 't4_3b'
MANUAL_PRESET = None

PRESETS = {
    # L4 22.5 GB (Ada, bf16): CodeLlama-7B 4-bit NF4
    'l4_7b': {
        'model_name': 'unsloth/codellama-7b-instruct-bnb-4bit',
        'max_seq_len': 2048,   # p95 примера ~830 токенов, max ~1500
        'batch_size': 8,
        'grad_accum': 2,       # effective batch = 16
    },
    # T4 15 GB и слабее: Qwen2.5-Coder-3B-Instruct 4-bit
    't4_3b': {
        'model_name': 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
        'max_seq_len': 2048,
        'batch_size': 8,
        'grad_accum': 2,
    },
}

if MANUAL_PRESET:
    PRESET_NAME = MANUAL_PRESET
else:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    PRESET_NAME = 'l4_7b' if vram_gb >= 20 else 't4_3b'
    print(f'GPU: {gpu_name} ({vram_gb:.1f} GB) -> preset {PRESET_NAME}')

CFG = PRESETS[PRESET_NAME]
MAX_SEQ_LEN = CFG['max_seq_len']
print('model:', CFG['model_name'], '| bs:', CFG['batch_size'], 'x ga:', CFG['grad_accum'])

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG['model_name'],
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

In [ ]:
# Датасет: keys = {id, instruction, output, metadata}
# text = instruction (CodeLlama [INST]-промпт, кончается [/INST]) + output (чистый JSON) + EOS
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return Dataset.from_list([json.loads(l) for l in f if l.strip()])

train_ds = load_jsonl(f'{DATA_DIR}/train.jsonl')
val_ds = load_jsonl(f'{DATA_DIR}/val.jsonl')
print('train:', len(train_ds), '| val:', len(val_ds))

EOS = tokenizer.eos_token

def format_example(rec):
    return {'text': rec['instruction'] + rec['output'] + EOS}

train_fmt = train_ds.map(format_example)
val_fmt = val_ds.map(format_example)
print(repr(train_fmt[0]['text'][-160:]))

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_fmt,
    eval_dataset=val_fmt,
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=CFG['batch_size'],
        gradient_accumulation_steps=CFG['grad_accum'],
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=2,
        per_device_eval_batch_size=16,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        weight_decay=0.01,
        seed=3407,
        output_dir='/content/checkpoints',
        report_to='none',
        max_seq_length=MAX_SEQ_LEN,
    ),
)

In [ ]:
# Маскирование промпта: loss только на JSON-комплишене.
# Вручную через offset mapping: train_on_responses_only ломается на свежих
# версиях unsloth (склейка токенов прячет маркер '[/INST]').
from transformers import DataCollatorForSeq2Seq

RESPONSE_MARKER = '[/INST]'

def tokenize_with_mask(rec):
    full = rec['text']
    enc = tokenizer(
        full,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_offsets_mapping=True,
    )
    idx = full.rfind(RESPONSE_MARKER)
    resp_start = (idx + len(RESPONSE_MARKER)) if idx != -1 else len(full)
    input_ids = enc['input_ids']
    labels = [
        (tid if b > resp_start else -100)
        for tid, (a, b) in zip(input_ids, enc['offset_mapping'])
    ]
    return {'input_ids': input_ids, 'labels': labels}

train_tok = train_fmt.map(
    tokenize_with_mask,
    remove_columns=train_fmt.column_names,
    desc='Tokenize + mask prompt',
)
val_tok = val_fmt.map(
    tokenize_with_mask,
    remove_columns=val_fmt.column_names,
    desc='Tokenize + mask prompt',
)

# Проверка: в каждом примере остались немаскированные токены
assert all(any(l != -100 for l in ex['labels']) for ex in train_tok.select(range(64)))

trainer.train_dataset = train_tok
# Eval на фиксированном подмножестве 200 примеров: полный val (850) на каждом
# eval добавлял ~2 ч на L4 без пользы для обучения. Полный val используется
# в ячейке JSON parse rate после тренинга.
trainer.eval_dataset = val_tok.select(range(min(200, len(val_tok))))
trainer.data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)
print('prompt-masked datasets ready:', len(train_tok), 'train,', len(trainer.eval_dataset), 'eval-subset')

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
# Быстрая проверка: генерация на валидационном примере (greedy)
FastLanguageModel.for_inference(model)

sample = val_ds[0]
inputs = tokenizer(sample['instruction'], return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)
generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('EXPECTED:', sample['output'])
print('GENERATED:', generated)

In [ ]:
# Метрика Phase 3: JSON parse rate на 64 val-примерах (порог цели >= 95%)
import json

def extract_json(s):
    a, b = s.find('{'), s.rfind('}')
    if a == -1 or b <= a:
        return None
    try:
        return json.loads(s[a:b + 1])
    except Exception:
        return None

ok = total = 0
for i in range(min(64, len(val_ds))):
    rec = val_ds[i]
    inputs = tokenizer(rec['instruction'], return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    total += 1
    ok += extract_json(gen) is not None

print(f'JSON parse rate: {ok}/{total} = {ok / total:.1%}')

In [ ]:
# Сохранить LoRA-адаптер на Drive (для inference.py)
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print('saved to', ADAPTER_OUT)

In [ ]:
# Опционально: слитая 16-bit модель для локального инференса
# (~7 GB для 7B, ~3 GB для 3B)
# model.save_pretrained_merged(ADAPTER_OUT + '_merged', tokenizer, save_method='merged_16bit')